# Day 14 — Klasik Görüntü Segmentasyonu
## Otsu Bimodal Eşikleme, Havza (Watershed) Segmentasyonu ve Bölgesel Kontur Analizi

> **Aşama:** Faz 2 — Bilgisayarlı Görü (Day 09–15)
> **Resmi Staj Defteri Konusu:** Klasik Görüntü Segmentasyonu (Yaprak 27 & 28)

### 1. Problem
Halı desenlerinin otomatik analizi ve kalite denetiminde, ana motiflerin (madalyon, bordür, geometrik şekiller) arka plan dokuma zemininden ayrıştırılması gerekir. Genel piksel yoğunluğu histogramları üzerinden doğru eşikleme yapılamadığında motif sınırları birbirine karışır.

### 2. Why the Problem Matters
Otsu algoritması sınıflar arası varyansı maksimize ederek bimodal histogramlardan otomatik eşik değeri çıkarır. Kontur analizi ise ayrıştırılan bölgelerin alan, çevre ve dairesellik (circularity) gibi geometrik özniteliklerini hesaplar.

### 3. Engineering Concepts
- **Otsu Eşikleme**: Sınıf içi varyansı minimize eden eşik değerinin $\sigma_B^2(t)$ maksimizasyonu ile bulunması.
- **Kontur Analizi (cv2.findContours)**: Sınır piksel zincirlerinin hiyerarşik vektörlerle temsil edilmesi.
- **Dairesellik (Circularity)**: $C = \frac{4\pi \cdot \text{Alan}}{\text{Çevre}^2}$ (Daire için 1.0, ince uzun yapılar için 0'a yakın).

In [ ]:
# 4. Library / API Investigation & Standalone Definitions
import cv2
import numpy as np
from typing import List, Tuple
from pydantic import BaseModel

class RegionProperties(BaseModel):
    region_id: int
    area_px: int
    perimeter_px: float
    circularity: float

class CarpetSegmenter:
    @staticmethod
    def segment_otsu(img: np.ndarray) -> Tuple[np.ndarray, float]:
        thresh, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return mask, float(thresh)

    @staticmethod
    def extract_regions(mask: np.ndarray) -> List[RegionProperties]:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        regions = []
        for i, cnt in enumerate(contours):
            area = cv2.contourArea(cnt)
            if area < 20:
                continue
            perimeter = cv2.arcLength(cnt, True)
            circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0.0
            regions.append(RegionProperties(
                region_id=i + 1,
                area_px=int(area),
                perimeter_px=round(float(perimeter), 1),
                circularity=round(float(circularity), 3)
            ))
        return regions

print("Segmentasyon Motoru Yüklendi.")


In [ ]:
# 5. Minimal Implementation
canvas = np.zeros((200, 200), dtype=np.uint8)
cv2.circle(canvas, (60, 60), 30, 200, -1)   # Dairesel motif
cv2.rectangle(canvas, (110, 110), (180, 180), 220, -1) # Dikdörtgen motif

mask, t_val = CarpetSegmenter.segment_otsu(canvas)
regions = CarpetSegmenter.extract_regions(mask)
print(f"Otsu Otomatik Eşiği: {t_val:.1f} | Tespit Edilen Motif Sayısı: {len(regions)}")
for r in regions:
    print(f"  Bölge {r.region_id}: Alan={r.area_px}px | Çevre={r.perimeter_px}px | Dairesellik={r.circularity:.3f}")

In [ ]:
# 6. Experiment: Geometrik Şekil Sınıflandırması
for r in regions:
    shape_type = "Dairesel Motif" if r.circularity > 0.75 else "Köşeli/Poligonal Motif"
    print(f"Bölge {r.region_id} Sınıfı -> {shape_type}")

In [ ]:
# 7. Visualization: Orijinal vs Segmentasyon Maskesi
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(canvas, cmap="gray")
axes[0].set_title("Orijinal Halı Deseni")
axes[0].axis("off")

axes[1].imshow(mask, cmap="viridis")
axes[1].set_title(f"Otsu İkili Maske (Eşik: {t_val:.0f})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert len(regions) == 2
assert any(r.circularity > 0.75 for r in regions)
print("Klasik segmentasyon ve kontur çıkarımı başarıyla doğrulandı.")

In [ ]:
# 9. Failure Cases: Tamamen siyah arka plan segmentasyonu
black = np.zeros((50, 50), dtype=np.uint8)
_, t_zero = CarpetSegmenter.segment_otsu(black)
no_regions = CarpetSegmenter.extract_regions(black)
assert len(no_regions) == 0
print("Boş görüntüde sıfır bölge tespiti doğrulandı.")

### 10. Conclusions
Otsu eşikleme ve kontur analiziyle halı motifleri arka plan dokusundan ayrıştırılmış, her bir bölgenin geometrik öznitelikleri sayısal olarak hesaplanmıştır.